# Bayes最適化　コード作成デモ<BR>
<div style="text-align: right;">
Nov. 2022<BR>
Masanobu NAKAYAMA<BR>
</div>


In [ ]:
#!pip install matplotlib
!wget https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part1/LCYZP_all.csv

In [ ]:
import math
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt


# 適当な関数によるベイズ最適化

ユーザーは関数x=f(x)を定義したけど、以後知らないふりをして、ベイズ最適化で当該関数の最大値 ([0,1]範囲)をなるべく少ない手数で調べる。

![fig001.png](https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part1/figs/fig001.png)


## データの定義・入力 (Numpy形式)

トレーニングデータ<BR>
観測点（ノイズあり）と真の関数 ( y = 3 sin(2πx) + 3 x   範囲[0,1] )を定義<BR>

In [ ]:
! pip install pandas matplotlib seaborn

In [ ]:
# トレーニングデータx (train_x)の作成　８点を乱数で選ぶ 範囲は[0,1]
np.random.seed(0)  # 乱数のシードを設定
train_x = np.random.rand(8)

#  トレーニングデータy (train_y)の作成  ８点　
#    関数は　 y = 3 * sin(2 * pi * x) + 3 * x  + rdm_y
#    ただしrdm_y は [-0.5, 0.5]の乱数（誤差）

rdm_y= np.random.rand(8)-0.5
train_y = 3*np.sin(train_x * (2 * math.pi)) + 3*train_x + rdm_y

plt.scatter(train_x,train_y)
print (train_x,"\n",train_y)

In [ ]:
#検証用　誤差のない真の関数　( y = 3 sin(2πx) + 3 x 範囲[0,1] )と、トレーニングデータを比較する。
# ベイズ最適化を行う際に直接必要な箇所ではない。

#  0～2の範囲を100点取り出す (true_x, true_y) 1～2の範囲は、後の過程で外挿空間の検証用に使用するため。
true_x=np.linspace(0,2,100)  
true_y = 3*np.sin(true_x * (2 * math.pi)) +3*true_x


#以下、train_x, train_yをプロット、　誤差のない真の関数true_x, true_yも実線描画

plt.xlim([0,1])
plt.ylim([-1,5])
plt.plot(true_x,true_y,linestyle='solid',color='red')  #理論曲線の描画
plt.scatter(train_x,train_y)  #サンプリングデータの描画



In [ ]:
train_x.shape

In [ ]:
#  train_x, train_yは横ベクトルだが、縦ベクトルにしないといけない。　変換 > train_nx,  train_ny

# train_x, train_yを縦ベクトルに変換
train_nx = train_x.reshape(-1, 1)  # 縦ベクトルに変換
train_ny = train_y.reshape(-1, 1)  # 縦ベクトルに変換

print(train_nx)
print(train_ny)

## ガウス過程回帰の実行

ガウス過程　回帰式はカーネルで定義  RBFカーネルがポピュラー<BR>
    
![fig002.png](https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part1/figs/fig002.png)    




![fig003.png](https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part1/figs/fig003.png)


In [ ]:
import sklearn.gaussian_process as gp
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process import kernels
from sklearn.preprocessing import StandardScaler

%matplotlib inline


ガウス過程回帰分析：　sklearnを今回は使用<BR>
取扱説明書  https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html
    
    gpm = gp.GaussianProcessRegressor()  でインスタンス化
    gpm.xxx() でフィッティング xxxは？
    gpm.xxx(x, return_std=True) は、作成した予測式をつかって予測および標準偏差を返す  xxxに入る言葉は？
    予測値をGPf,  標準偏差をGPsd としたとき、どのように値を回収するか？

In [ ]:
kk=gp.kernels.ConstantKernel()*gp.kernels.RBF() + gp.kernels.WhiteKernel()
#kk=gp.kernels.RBF(length_scale=2) 
#kk=gp.kernels.Matern(nu=5)
#kk=gp.kernels.Matern(nu=5) + gp.kernels.WhiteKernel()

#回帰条件の設定  gpm　としてインスタンス化
gpm = GaussianProcessRegressor(kernel=kk, alpha=0.1, n_restarts_optimizer=10, normalize_y=True)

#gpmを使ってフィッティング gpm.***(***,***)
gpm.fit(train_nx, train_ny) 

#　フィッティングの良さは、対数尤度(log-likelihood)によってあらわされる（この値が大きくなるように最適化される）

print(gpm.log_marginal_likelihood())

print ("theta0",gpm.kernel.theta)   #チューニング前
print ("theta1",gpm.kernel_.theta)   #チューニング後
print(gpm.kernel.bounds)


In [ ]:
print(gpm.predict(train_nx))
print(train_ny)
plt.scatter(train_ny, gpm.predict(train_nx))
plt.show()


In [ ]:
newx = np.linspace(0,2,100).reshape(100,-1)
GPf, GPsd = gpm.predict(newx, return_std=True)  # 予測値(GPf)と標準偏差(GPsd)

# なぜかGPf（縦ベクトル）とGPsd（横ベクトル）の出力形式がちがうので、行列を縦ベクトル化
# pythonのバージョンが3.9以下であれば、GPfのreshape処理は不要
GPf=GPf.reshape(-1,1)
GPsd=GPsd.reshape(-1,1)

回帰分析結果の表示　matplotlib <BR>

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(8, 6))
    
# Get upper and lower confidence bounds
lower = (GPf-GPsd)#.flatten()
upper = (GPf+GPsd)#.flatten()

# Plot training data as black stars
ax.plot(train_nx, train_ny, 'kx')
# Plot predictive means as blue line
ax.plot(newx, GPf, 'blue')
# Shade between the lower and upper confidence bounds
ax.fill_between(newx.flatten(),lower.flatten(), upper.flatten(),alpha=0.3)
ax.set_xlim([0,1.5])
ax.set_ylim([-2, 8])
ax.plot(true_x,true_y,linestyle='dotted',color='red')
ax.legend(['Observed Data', 'Mean', 'Confidence','true'])
print ("gpm.kernel.theta",gpm.kernel.theta)
print ("gpm.kernel",gpm.kernel)

plt.show()

![fig004.png](https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day5_Part1/figs/fig004.png)
)

In [ ]:
newx

In [ ]:
#　以下のEI値は最大値を探すように定義されている。最小値を探したい場合はどうすればよいか？

from scipy.stats import norm

current_ymax=np.max(train_ny)
Z = (GPf-current_ymax)/GPsd
ei = ((GPf-current_ymax)*norm.cdf(Z))+(GPsd*norm.pdf(Z))

plt.plot(newx,ei)     #  xに対するeiの変化を示せ
plt.show()




■　問題

xを[0,1]に限定して次に評価すべきx値は？　また予想されるyの値は？<BR> 
    
ヒント<BR>
np.max(a): a中の最大値を出力 <BR>
np.argmax(a):  a中で最大値となるindexを出力<BR>
a[0:49]: index 0 ～ 49　を指定 <BR>

In [ ]:
print ("next_x:",np.argmax(ei[0:49]))
print(newx[14])
print ("max_ei:",ei[np.argmax(ei[0:49])])
print ("expected_y:", GPf[np.argmax(ei[0:49])])

### 演習：　
以下に定義したblackbox関数（numpy行列 train_nx, train_ny を入力したら、ベイズ最適化をして次に調査すべきnxを出力する）を使って、yの最大値を調査するプログラムを作成せよ。なお、最初にユーザーがx1stで入力した値を初期値とする。

### ⚠️ 注意：以下の`blackbox`関数には問題があります
EIが最大となる点に毎回収束してしまい、同じ点を繰り返し探索するという問題があります。
次のセルで実際にその挙動を確認し、その後の改良版と比較してください。

In [ ]:
def blackbox(train_nx,train_ny):
    
    kk=gp.kernels.ConstantKernel()*gp.kernels.RBF() + gp.kernels.WhiteKernel()
    gpm = gp.GaussianProcessRegressor(kernel=kk, alpha=0.1, n_restarts_optimizer=30, normalize_y=True)
    gpm.fit(train_nx, train_ny) 
    #print("Likelihood:",gpm.log_marginal_likelihood())
    newx = np.linspace(0,1,100).reshape(100,-1)
    GPf, GPsd = gpm.predict(newx, return_std=True) 
    GPf=GPf.reshape(-1,1)
    GPsd=GPsd.reshape(-1,1)
    current_ymax=np.max(train_ny)
    Z = (GPf-current_ymax)/GPsd
    ei = ((GPf-current_ymax)*norm.cdf(Z))+(GPsd*norm.pdf(Z))
    nx=newx[np.argmax(ei)][0]

    return nx


In [ ]:
x1st= 0.8   #　適当に最初のxを指定する [0,1]

x=np.array([x1st])
for i in range(20): #20回目で当てられるか？？

    rdm=np.random.rand(1)-0.5
    y = 3*np.sin(x * (2 * math.pi)) + 3*x + rdm
    
    nx=blackbox(x.reshape(-1,1),y.reshape(-1,1))  #blackbox関数を使って、次のx値を決定(nx)　（注）縦ベクトル変換すること
    #nxの値をxベクトルに追加
    x=np.append(x,nx)  #nxの値をxベクトルに追加
    print ("nx", nx)


### ✅ 改良版`blackbox`関数
既探索点と重複しないよう、EI上位点から順に未探索の点を選ぶように修正した版です。

In [ ]:
import numpy as np
import math
from scipy.stats import norm
import sklearn.gaussian_process as gp

def blackbox(train_nx, train_ny, tried_x, tol=1e-4):
    kk = gp.kernels.ConstantKernel() * gp.kernels.RBF() + gp.kernels.WhiteKernel()
    gpm = gp.GaussianProcessRegressor(kernel=kk, alpha=0.1, n_restarts_optimizer=30, normalize_y=True)
    gpm.fit(train_nx, train_ny)

    newx = np.linspace(0, 1, 100).reshape(-1, 1)
    GPf, GPsd = gpm.predict(newx, return_std=True)
    GPf = GPf.reshape(-1, 1)
    GPsd = GPsd.reshape(-1, 1)
    current_ymax = np.max(train_ny)

    Z = (GPf - current_ymax) / GPsd
    ei = ((GPf - current_ymax) * norm.cdf(Z)) + (GPsd * norm.pdf(Z))

    # ei を降順にソートし、未探索の点から選ぶ
    sorted_indices = np.argsort(-ei.flatten())  # 降順
    for idx in sorted_indices:
        candidate = newx[idx][0]
        if not np.any(np.abs(tried_x - candidate) < tol):
            return candidate

    # fallback（すべて近すぎる場合）
    return newx[sorted_indices[0]][0]

# 初期点
x1st = 0.8
x = np.array([x1st])
y = []

for i in range(20):
    rdm = np.random.rand(1) - 0.5
    y_val = 3 * np.sin(x[-1] * (2 * math.pi)) + 3 * x[-1] + rdm
    y.append(y_val)

    nx = blackbox(x.reshape(-1, 1), np.array(y).reshape(-1, 1), tried_x=x)
    x = np.append(x, nx)
    print(f"{i+1:02d}回目: nx = {nx:.5f}")


In [ ]:
plt.plot(y)   #yの値の変化をプロット（時系列）
plt.show()

In [ ]:
plt.plot(x)   #xの値の変化をプロット（時系列）
plt.show()



# ここから、論文データを用いたベイズ最適化例

<BR>  データ引用元：　K. Nakano, Y. Noda, N. Tanibata, H. Takeda, M. Nakayama, R. Kobayashi, I. Takeuchi, "Exhaustive and Informatics-Aided Search for Fast Li-Ion Conductor with NASICON-Type Structure Using Material Simulation and Bayesian Optimization", APL Materials,  8, 041112 (2020)　https://doi.org/10.1063/5.0007414
    

１．データの呼び込み  csv形式データを true_x, true_yに格納　　(true_xは２次元, true_yは１次元）

In [ ]:
path=str("./LCYZP_all.csv")
df_original=pd.read_csv(path, index_col=0)
df = df_original.dropna()
drop_columns = ['s@300','logs@300']
df_descriptor = df.drop(drop_columns, axis=1)
target_columns = ['logs@300']
df_target = df[target_columns]

true_x=df_descriptor.to_numpy()
true_y=df_target.to_numpy()

print ("max(y)",np.max(true_y))

In [ ]:
true_y

２．読み込んだデータを２次元表示

In [ ]:
#  参考　プロットの色を物性値(z)にリンクさせて x, y, zプロット。

from matplotlib import cm

plt.figure(figsize=[5,4],dpi=150)
plt.scatter(true_x[:,0],true_x[:,1],c=true_y,cmap=cm.jet)
ax=plt.colorbar()#カラーマップの凡例
ax.set_label('log(sigma(300K))')#カラーバーのラベルネーム
plt.show()



３．データセットから６点をランダムに抽出（初期トレーニングデータとする）

In [ ]:
train_x=np.array([])
train_y=np.array([])

for i in range(6):
    iadd=np.random.randint(len(true_y))
    train_x=np.append(train_x,true_x[iadd,:])
    train_y=np.append(train_y,true_y[iadd,:])
    true_x=np.delete(true_x,iadd,0)
    true_y=np.delete(true_y,iadd,0)

train_x=train_x.reshape(-1,3)
train_y=train_y.reshape(-1,1)

print(train_x)
print(train_y)

In [ ]:
# グラフ上にサンプリングした６点を表示

plt.figure(figsize=[5,4],dpi=150)
plt.scatter(train_x[:,0],train_x[:,1],c=train_y,cmap=cm.jet)
ax=plt.colorbar()#カラーマップの凡例
ax.set_label('log(sigma(300K))')#カラーバーのラベルネーム
plt.show()


４．ガウス過程回帰をして、GPf, GPsdを算出 (注：GPf, GPsdはtrue_xに対して適用すること)

In [ ]:
GPf=np.array([])
GPv=np.array([])
Gpsd=np.array([])
GPtry=np.array([])

kk= gp.kernels.ConstantKernel()*gp.kernels.RBF() + gp.kernels.WhiteKernel()   #カーネル関数など定義
gpm= GaussianProcessRegressor(kernel=kk, alpha=0.1, n_restarts_optimizer=30, normalize_y=True)   #ガウス過程のインスタンス化
gpm.fit(train_x,train_y)                                     #ガウス過程のフィッティング

GPf, GPsd = gpm.predict(true_x, return_std=True)             #予測式を用いて true_xの値から GPf, GPsdを予測
GPf=GPf.reshape(-1,1)
GPsd = GPsd.reshape(-1,1)                   # 縦ベクトル変換

plt.scatter(true_y,GPf)
plt.show()

In [ ]:
ei=np.array([])
current_ymax=np.max(train_y)
Z = (GPf-current_ymax)/GPsd
ei = ((GPf-current_ymax)*norm.cdf(Z))+(GPsd*norm.pdf(Z))

print ("max ei:",np.max(ei))
print ("next x:",true_x[np.argmax(ei)])

In [ ]:

plt.figure(figsize=[5,4],dpi=150)
plt.scatter(true_x[:,0],true_x[:,1],c=ei,cmap=cm.jet)
ax=plt.colorbar()#カラーマップの凡例
ax.set_label('ei')#カラーバーのラベルネーム
plt.show()
